In [3]:
!pip install pandas
!pip install scikit-learn
!pip install matplotlib

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ----------- ---------------------------- 2.9/9.7 MB 17.8 MB/s eta 0:00:01
   ------------------------------------ --- 8.9/9.7 MB 23.8 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 22.9 MB/s  0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ------------------------ --------------- 7.6/12.3 MB 36.3 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 34.3 MB/s  0:00:00

   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -----------

In [4]:
import pandas as pd

df_racism = pd.read_csv("Dataset/RacismDetectionDataSet.csv")

# rename for consistency
df_racism = df_racism.rename(columns={
    "Comment": "text",
    "Label": "label"
})

df_racism.head()

,text,label
0,i was born a racist and I will die a racist I ...,1
1,bitch nigga miss me with that,1
2,if you aint bout that murder game pussy nigga ...,1
3,gay niggas couldnt wait to act like bitches to...,1
4,why deos a gorilla always have a frown because...,1


In [5]:
import re

def clean_text(text):

    text = str(text)

    # remove urls
    text = re.sub(r"http\S+|www\S+", "", text)

    # remove mentions
    text = re.sub(r"@\w+", "", text)

    # remove html entities
    text = re.sub(r"&amp;", "and", text)

    # remove weird unicode
    text = text.encode("ascii", "ignore").decode()

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

df_racism["clean_text"] = df_racism["text"].apply(clean_text)

df_racism = df_racism[["clean_text","label"]]

In [6]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_racism,
    test_size=0.2,          # 80/20 split
    stratify=df_racism["label"],
    random_state=42
)

print("Train:", len(train_df))
print("Test:", len(test_df))

print(train_df["label"].value_counts(normalize=True))
print(test_df["label"].value_counts(normalize=True))

Train: 1599
Test: 400
label
1    0.500313
0    0.499687
Name: proportion, dtype: float64
label
1    0.5
0    0.5
Name: proportion, dtype: float64


In [7]:
train_df.to_csv("racism_train.csv", index=False)
test_df.to_csv("racism_test.csv", index=False)

In [22]:
sex_train = pd.read_csv("Dataset/train.csv")
sex_test = pd.read_csv("Dataset/test.csv")

racism_train = pd.read_csv("Dataset/racism_train.csv")
racism_test = pd.read_csv("Dataset/racism_test.csv") 

In [23]:
def process_sexism(df):

    df = df[["text", "label_sexist"]]

    df = df.rename(columns={
        "label_sexist": "sexism_label"
    })

    # convert labels
    df["sexism_label"] = df["sexism_label"].map({
        "sexist": 1,
        "not sexist": 0
    })

    # racism label not available
    df["racism_label"] = -1

    return df

sex_train = process_sexism(sex_train)
sex_test = process_sexism(sex_test)

In [24]:
def process_racism(df):

    df = df.rename(columns={
        "clean_text": "text",
        "label": "racism_label"
    })

    df = df[["text", "racism_label"]]

    # sexism label not available
    df["sexism_label"] = -1

    return df

racism_train = process_racism(racism_train)
racism_test = process_racism(racism_test)

In [25]:
def reorder(df):

    return df[["text", "sexism_label", "racism_label"]]

sex_train = reorder(sex_train)
sex_test = reorder(sex_test)

racism_train = reorder(racism_train)
racism_test = reorder(racism_test)

In [26]:
train_data = pd.concat(
    [sex_train, racism_train],
    ignore_index=True
)

test_data = pd.concat(
    [sex_test, racism_test],
    ignore_index=True
)


train_data.head()

,text,sexism_label,racism_label
0,"Then, she's a keeper. 😉",0,-1
1,This is like the Metallica video where the poo...,0,-1
2,woman?,0,-1
3,Unlicensed day care worker reportedly tells co...,0,-1
4,[USER] Leg day is easy. Hot girls who wear min...,1,-1


In [27]:
train_data.to_csv("bias_train.csv", index=False)
test_data.to_csv("bias_test.csv", index=False)